# BloodMNIST Experiments (CNN or MLP)
Use this notebook as the runner. Shared code lives in the `ml/` package.

In [31]:
from ml.data import build_bloodmnist_dataloaders
from ml.models import CNN, MLP
from ml.setup import get_device, make_criterion, make_optimizer
from ml.training import EarlyStopping, train_with_validation, evaluate_accuracy

In [32]:
BATCH_SIZE = 128
data_bundle = build_bloodmnist_dataloaders(batch_size=BATCH_SIZE, download=True)

train_loader = data_bundle['train_loader']
val_loader = data_bundle['val_loader']
test_loader = data_bundle['test_loader']
n_channels = data_bundle['n_channels']
n_classes = data_bundle['n_classes']

print(f'n_channels={n_channels}, n_classes={n_classes}')

n_channels=3, n_classes=8


In [ ]:
MODEL_NAME = 'mlp'   # choose: 'cnn' or 'mlp'
OPTIMIZER_NAME = 'adam'  # choose: 'adam' or 'sgd'
LEARNING_RATE = 0.0001   # 0.01 for sgd
MOMENTUM = 0.9
DROPOUT = 0.3
PATIENCE = 50
EARLY_STOPIING_DELTA = 0.0002

device = get_device()

if MODEL_NAME == 'cnn':
    model = CNN(in_channels=n_channels, num_classes=n_classes, dropout=DROPOUT).to(device)
elif MODEL_NAME == 'mlp':
    model = MLP(input_dim=n_channels * 28 * 28, num_classes=n_classes, dropout=DROPOUT).to(device)
else:
    raise ValueError('MODEL_NAME must be cnn or mlp')

criterion = make_criterion('cross_entropy')
optimizer = make_optimizer(model, name=OPTIMIZER_NAME, lr=LEARNING_RATE, momentum=MOMENTUM)
early_stopping = EarlyStopping(PATIENCE, EARLY_STOPIING_DELTA)

print(f'Device: {device}')
print(f'Model: {MODEL_NAME}')
print(f'Optimizer: {OPTIMIZER_NAME}, lr={LEARNING_RATE}')

Device: cpu
Model: mlp
Optimizer: adam, lr=0.0001


In [ ]:
history = train_with_validation(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=350,
    early_stopping=early_stopping,
)

print('Stopped epoch:', history['stopped_epoch'])
print('Best epoch:', history['best_epoch'])

Epoch 1	Training Loss: 1.8515	Validation Loss: 1.5783
Epoch 2	Training Loss: 1.4839	Validation Loss: 1.2949
Epoch 3	Training Loss: 1.2829	Validation Loss: 1.1128
Epoch 4	Training Loss: 1.1408	Validation Loss: 0.9963
Epoch 5	Training Loss: 1.0514	Validation Loss: 0.9228
Epoch 6	Training Loss: 0.9904	Validation Loss: 0.8657
Epoch 7	Training Loss: 0.9513	Validation Loss: 0.8227
Epoch 8	Training Loss: 0.9203	Validation Loss: 0.7886
Epoch 9	Training Loss: 0.8895	Validation Loss: 0.7735
Epoch 10	Training Loss: 0.8612	Validation Loss: 0.7510
Epoch 11	Training Loss: 0.8432	Validation Loss: 0.7290
Epoch 12	Training Loss: 0.8274	Validation Loss: 0.7284
Epoch 13	Training Loss: 0.8086	Validation Loss: 0.7056
Epoch 14	Training Loss: 0.7907	Validation Loss: 0.6911
Epoch 15	Training Loss: 0.7801	Validation Loss: 0.6723
Epoch 16	Training Loss: 0.7640	Validation Loss: 0.6795
Epoch 17	Training Loss: 0.7452	Validation Loss: 0.6506
Epoch 18	Training Loss: 0.7382	Validation Loss: 0.6431
Epoch 19	Training L

In [35]:
test_accuracy = evaluate_accuracy(model, test_loader, device)
print(f'Test accuracy: {test_accuracy:.2f}%')

Test accuracy: 87.23%
